# Pipeline de Dados em Nuvem — ETL com base real

Este notebook implementa um fluxo ETL (*Extract, Transform, Load*) usando Python e Pandas, agora aplicado à base pública **Online Retail**, disponibilizada pelo UCI Machine Learning Repository.

**Atenção:** o processamento é executado no ambiente do Colab/Jupyter e os resultados são salvos em arquivos CSV e JSON. O pipeline incorpora testes automatizados, monitoramento de qualidade, rastreabilidade da execução e armazenamento em camadas. Quando executado no Google Colab com o Google Drive habilitado, os arquivos RAW, tratados e relatórios são persistidos no Drive, permanecendo disponíveis após o encerramento da sessão. Fora do Colab, o notebook utiliza armazenamento local.

## Fonte dos dados
Chen, D. (2015). *Online Retail*. UCI Machine Learning Repository. https://doi.org/10.24432/C5BW33

A base reúne 541.909 linhas de transações de uma empresa varejista on-line do Reino Unido, referentes ao período de 1º de dezembro de 2010 a 9 de dezembro de 2011. A licença informada pelo repositório é CC BY 4.0.

## Regras de tratamento
- Padronizar nomes de colunas e tipos;
- Remover linhas integralmente duplicadas;
- Desconsiderar faturas de cancelamento (identificadas pelo prefixo `C`);
- Descartar registros sem data, código/nome de produto ou quantidade/preço válidos;
- Descartar quantidades e preços não positivos;
- Calcular o valor bruto da linha (`quantidade × preço unitário`);
- Preservar país e identificador de cliente quando disponíveis.

O identificador da fatura é mantido como texto, pois pode conter letras nos registros de cancelamento.


## 1. Preparação do ambiente e conexão com a nuvem


In [ ]:
# Instalação das dependências
%pip -q install ucimlrepo openpyxl

from pathlib import Path
import json
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Persistência em nuvem:
# No Google Colab, monte o Google Drive e armazene o projeto nele.
# Fora do Colab, utilize uma pasta local como alternativa.
USE_GOOGLE_DRIVE = True

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/pipeline-dados-nuvem')
    STORAGE_MODE = 'Google Drive'
else:
    BASE_DIR = Path('./pipeline-dados-nuvem')
    STORAGE_MODE = 'Armazenamento local'

RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'outputs'

for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Modo de armazenamento:', STORAGE_MODE)
print('Diretório principal:', BASE_DIR.resolve())
print('Pastas preparadas com sucesso.')


## 2. Extração — obtenção da base real

In [ ]:
# Busca a base diretamente no UCI Machine Learning Repository (dataset id 352).
online_retail = fetch_ucirepo(id=352)

# O conjunto original possui variáveis de transação e metadados.
df = online_retail.data.original.copy()

print('Dimensão da base original:', df.shape)
print('Colunas:', df.columns.tolist())
display(df.head())


In [ ]:
# Camada RAW: preserva uma cópia da fonte antes de qualquer tratamento.
# O arquivo permite auditoria e reprocessamento sem depender de uma nova extração.
raw_file = RAW_DIR / 'online_retail_original.csv'
df.to_csv(raw_file, index=False, encoding='utf-8-sig')
print('Cópia bruta salva em:', raw_file)


## 3. Transformação e validação

In [ ]:
rows_input = len(df)

# Padronização dos nomes das colunas
df.columns = (
    df.columns.astype(str).str.strip().str.lower()
      .str.replace(' ', '_', regex=False)
)

required = ['invoiceno', 'stockcode', 'description',
            'quantity', 'invoicedate', 'unitprice']
missing_columns = [c for c in required if c not in df.columns]
if missing_columns:
    raise ValueError(f'Colunas obrigatórias ausentes na fonte: {missing_columns}')

# Conversão de tipos
df['invoiceno'] = df['invoiceno'].astype('string').str.strip()
df['stockcode'] = df['stockcode'].astype('string').str.strip()
df['description'] = df['description'].astype('string').str.strip()
df['invoicedate'] = pd.to_datetime(df['invoicedate'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unitprice'] = pd.to_numeric(df['unitprice'], errors='coerce')

# Remoção de duplicatas exatas
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

# Cancelamentos: número da fatura iniciado por C
cancellation_mask = df['invoiceno'].str.upper().str.startswith('C', na=False)
cancellations_removed = int(cancellation_mask.sum())
df = df.loc[~cancellation_mask].copy()

# Regras de validade para as linhas de venda
valid_mask = (
    df['invoiceno'].notna() & df['invoiceno'].ne('') &
    df['stockcode'].notna() & df['stockcode'].ne('') &
    df['description'].notna() & df['description'].ne('') &
    df['invoicedate'].notna() &
    df['quantity'].notna() & df['quantity'].gt(0) &
    df['unitprice'].notna() & df['unitprice'].gt(0)
)

invalid_rows = int((~valid_mask).sum())
clean = df.loc[valid_mask].copy()

# Esquema de saída organizado para análise de vendas
rename_map = {
    'invoiceno': 'id_venda',
    'stockcode': 'codigo_produto',
    'description': 'produto',
    'invoicedate': 'data',
    'quantity': 'quantidade',
    'unitprice': 'valor_unitario',
    'customerid': 'id_cliente',
    'country': 'pais'
}
clean = clean.rename(columns=rename_map)
clean['valor_total'] = (clean['quantidade'] * clean['valor_unitario']).round(2)
clean['data'] = clean['data'].dt.strftime('%Y-%m-%d %H:%M:%S')

# Seleciona as colunas disponíveis, preservando cliente e país se existirem
output_columns = [
    'id_venda', 'data', 'codigo_produto', 'produto', 'quantidade',
    'valor_unitario', 'valor_total', 'id_cliente', 'pais'
]
output_columns = [c for c in output_columns if c in clean.columns]
clean = clean[output_columns].reset_index(drop=True)

print('Registros após tratamento:', len(clean))
display(clean.head(10))


## 4. Testes automatizados de qualidade

Os testes abaixo verificam regras essenciais em uma amostra pequena e também conferem invariantes da base completa. Se uma regra deixar de funcionar, a execução é interrompida com erro.

In [ ]:
# Testes unitários simples das regras de negócio com dados controlados.
import pandas as pd

mini = pd.DataFrame({
    'id_venda': ['100', '101', 'C102', '103'],
    'data': ['2011-01-01', '2011-01-02', '2011-01-03', '2011-01-04'],
    'codigo_produto': ['A', 'B', 'C', 'D'],
    'produto': ['Produto A', 'Produto B', 'Produto C', 'Produto D'],
    'quantidade': [2, 0, 1, 3],
    'valor_unitario': [10.0, 5.0, 8.0, 2.0]
})
mini['data'] = pd.to_datetime(mini['data'], errors='coerce')
mask_teste = (
    mini['id_venda'].notna() & ~mini['id_venda'].str.startswith('C') &
    mini['data'].notna() & mini['codigo_produto'].notna() &
    mini['produto'].notna() & mini['quantidade'].gt(0) &
    mini['valor_unitario'].gt(0)
)
mini_valida = mini.loc[mask_teste].copy()
assert len(mini_valida) == 2, 'Falha: regra de validade/cancelamento.'
assert (mini_valida['quantidade'] * mini_valida['valor_unitario']).tolist() == [20.0, 6.0], 'Falha no cálculo do total.'

# Invariantes da base tratada
assert clean['id_venda'].notna().all(), 'Existem vendas sem identificador.'
assert clean['quantidade'].gt(0).all(), 'Existem quantidades não positivas.'
assert clean['valor_unitario'].gt(0).all(), 'Existem preços não positivos.'
assert clean['valor_total'].ge(0).all(), 'Existem valores totais negativos.'
assert not clean.duplicated().any(), 'Existem linhas duplicadas na saída.'
print('Todos os testes automatizados foram aprovados.')


## 5. Carga, monitoramento e rastreabilidade

In [ ]:
import hashlib
from datetime import datetime, timezone
import platform

output_file = PROCESSED_DIR / 'online_retail_tratado.csv'
report_file = OUTPUT_DIR / 'relatorio_qualidade.json'
lineage_file = OUTPUT_DIR / 'data_lineage.json'

# Hash da entrada para identificar exatamente a versão dos dados processados.
input_hash = hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()
pipeline_version = '1.1.0'

# Idempotência: se o CSV existente tiver o mesmo conteúdo, evita regravá-lo.
csv_content = clean.to_csv(index=False, encoding='utf-8-sig')
output_hash = hashlib.sha256(csv_content.encode('utf-8-sig')).hexdigest()
if output_file.exists():
    existing_hash = hashlib.sha256(output_file.read_bytes()).hexdigest()
    if existing_hash == output_hash:
        print('Saída já está atualizada; gravação ignorada (execução idempotente).')
    else:
        clean.to_csv(output_file, index=False, encoding='utf-8-sig')
else:
    clean.to_csv(output_file, index=False, encoding='utf-8-sig')

# Monitoramento: alerta quando 40% ou mais das linhas de entrada não chegam à saída.
discarded_total = rows_input - len(clean)
discarded_pct = (discarded_total / rows_input * 100) if rows_input else 0.0
threshold_pct = 40.0
quality_alert = discarded_pct >= threshold_pct
if quality_alert:
    print(f'ALERTA DE QUALIDADE: {discarded_pct:.2f}% dos registros não chegaram à saída (limite: {threshold_pct:.0f}%).')
else:
    print(f'Monitoramento OK: {discarded_pct:.2f}% dos registros não chegaram à saída.')

run_time = datetime.now(timezone.utc).astimezone().isoformat(timespec='seconds')
report = {
    'status': 'alerta' if quality_alert else 'sucesso',
    'fonte': 'UCI Machine Learning Repository — Online Retail (id 352)',
    'versao_pipeline': pipeline_version,
    'data_execucao': run_time,
    'registros_entrada': int(rows_input),
    'duplicatas_removidas': int(duplicates_removed),
    'cancelamentos_removidos': int(cancellations_removed),
    'registros_invalidos_descartados': int(invalid_rows),
    'registros_saida': int(len(clean)),
    'percentual_descartado': round(discarded_pct, 2),
    'limite_alerta_percentual': threshold_pct,
    'alerta_qualidade': bool(quality_alert),
    'percentual_retido_sobre_entrada': round(len(clean) / rows_input * 100, 2) if rows_input else 0.0,
    'periodo_data_minima': str(clean['data'].min()) if len(clean) else None,
    'periodo_data_maxima': str(clean['data'].max()) if len(clean) else None,
    'valor_total_vendas': round(float(clean['valor_total'].sum()), 2) if len(clean) else 0.0,
    'hash_entrada_sha256': input_hash,
    'hash_saida_sha256': output_hash,
    'arquivo_bruto': str(raw_file),
    'arquivo_saida': str(output_file),
    'colunas_saida': list(clean.columns)
}
with open(report_file, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# Data lineage: registra origem, transformação, destino e ambiente de execução.
lineage = {
    'pipeline': 'ETL Online Retail',
    'versao': pipeline_version,
    'executado_em': run_time,
    'origem': 'UCI Machine Learning Repository — dataset 352',
    'entrada': {'arquivo': str(raw_file), 'sha256_dataframe': input_hash, 'registros': int(rows_input)},
    'transformacoes': ['padronizacao de colunas e tipos', 'remocao de duplicatas', 'remocao de cancelamentos', 'validacao de campos e valores', 'calculo do valor_total'],
    'saida': {'arquivo': str(output_file), 'sha256_csv': output_hash, 'registros': int(len(clean))},
    'relatorio_qualidade': str(report_file),
    'ambiente': {'python': platform.python_version(), 'pandas': pd.__version__}
}
with open(lineage_file, 'w', encoding='utf-8') as f:
    json.dump(lineage, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print('CSV tratado:', output_file)
print('Relatório de qualidade:', report_file)
print('Rastreabilidade:', lineage_file)


## 6. Indicadores descritivos

In [ ]:
if len(clean):
    print('Quantidade de linhas tratadas:', len(clean))
    print('Faturas distintas:', clean['id_venda'].nunique())
    print('Produtos distintos:', clean['codigo_produto'].nunique())
    if 'pais' in clean.columns:
        print('Países distintos:', clean['pais'].nunique())
    print('Valor bruto total (GBP):', round(clean['valor_total'].sum(), 2))
    display(clean.groupby('pais', dropna=False)['valor_total']
                 .sum().sort_values(ascending=False).head(10).reset_index()
                 if 'pais' in clean.columns else clean.head())
else:
    print('Nenhum registro válido após as regras de tratamento.')


## 7. Persistência em nuvem — Google Drive

Nesta etapa, os arquivos gerados pelo pipeline são mantidos no Google Drive quando o notebook é executado no Google Colab e a opção `USE_GOOGLE_DRIVE` está habilitada. Como o diretório principal (`BASE_DIR`) foi definido dentro da pasta `MyDrive`, as camadas `raw`, `processed` e `outputs` são gravadas diretamente no armazenamento em nuvem, sem depender da vida útil da sessão do Colab.

A verificação abaixo confirma a existência dos principais arquivos e apresenta seus tamanhos. A integração utiliza o Drive montado pelo próprio usuário; não é necessário inserir chaves de API ou credenciais no notebook.


In [ ]:
# Verifica os arquivos persistidos no diretório configurado.
cloud_files = [raw_file, output_file, report_file, lineage_file]

storage_status = []
for file_path in cloud_files:
    exists = file_path.exists()
    size_kb = round(file_path.stat().st_size / 1024, 2) if exists else None
    storage_status.append({
        'arquivo': str(file_path.relative_to(BASE_DIR)),
        'disponivel': exists,
        'tamanho_kb': size_kb
    })

storage_report = pd.DataFrame(storage_status)
display(storage_report)

if not all(item['disponivel'] for item in storage_status):
    raise FileNotFoundError('Um ou mais arquivos esperados não foram encontrados no armazenamento.')

print(f'Persistência verificada em: {STORAGE_MODE}')
print(f'Diretório de armazenamento: {BASE_DIR.resolve()}')


## 8. Download dos resultados (Google Colab)

In [ ]:
try:
    from google.colab import files
    files.download(str(output_file))
    files.download(str(report_file))
    files.download(str(lineage_file))
except ImportError:
    print('Fora do Google Colab. Os arquivos estão salvos em:', PROCESSED_DIR, 'e', OUTPUT_DIR)


O pipeline registra volumes de entrada e saída, descartes, indicadores de vendas, hashes dos dados e metadados de execução. Os testes automatizados verificam regras de negócio e invariantes da saída; o monitoramento emite alerta quando o percentual descartado atinge 40%. A rastreabilidade é registrada em `data_lineage.json`, permitindo relacionar fonte, versão do código, transformação e arquivos gerados.

No Google Colab, com `USE_GOOGLE_DRIVE = True`, os arquivos são gravados diretamente no Google Drive, organizados nas camadas `data/raw`, `data/processed` e `outputs`. A etapa de persistência verifica a existência dos arquivos armazenados. Fora do Colab, o notebook utiliza uma pasta local. Esta implementação utiliza o Google Drive como armazenamento persistente; não implementa Airflow, banco de dados gerenciado, alertas por e-mail, controle de acesso ou processamento distribuído.